# CELDA 1 — CONFIGURACIÓN
Cambia JORNADA y vuelve a ejecutar solo la Celda 4 para simular la siguiente jornada sin reiniciar el cluster


In [ ]:
CONNECTION_STRING = "connection_string_del_eventhub"
EVENT_HUB_NAME    = "nombre_del_eventhub"   # empieza por es_
 
JORNADA         = 1      # ← cambia este número para cada jornada
DELAY_SEGUNDOS  = 0.05    # segundos entre eventos (0.2 = rápido, 1.5 = lento)
 
print(f"✅ Config cargada | Jornada: {JORNADA} | Delay: {DELAY_SEGUNDOS}s")

#  CELDA 2 — IMPORTS + DATOS SINTÉTICOS


In [ ]:
%pip install azure-eventhub

In [ ]:
import json, time, random, datetime
from dataclasses import dataclass, asdict
from typing import Optional
from azure.eventhub import EventHubProducerClient, EventData

random.seed(42)

# ── Equipos ──────────────────────────────────────────────────
EQUIPOS = [
    {"id":  1, "nombre": "Real Madrid",      "ciudad": "Madrid"},
    {"id":  2, "nombre": "FC Barcelona",     "ciudad": "Barcelona"},
    {"id":  3, "nombre": "Atlético Madrid",  "ciudad": "Madrid"},
    {"id":  4, "nombre": "Sevilla FC",       "ciudad": "Sevilla"},
    {"id":  5, "nombre": "Real Betis",       "ciudad": "Sevilla"},
    {"id":  6, "nombre": "Valencia CF",      "ciudad": "Valencia"},
    {"id":  7, "nombre": "Athletic Club",    "ciudad": "Bilbao"},
    {"id":  8, "nombre": "Real Sociedad",    "ciudad": "San Sebastián"},
    {"id":  9, "nombre": "Villarreal CF",    "ciudad": "Villarreal"},
    {"id": 10, "nombre": "Getafe CF",        "ciudad": "Getafe"},
    {"id": 11, "nombre": "Osasuna",          "ciudad": "Pamplona"},
    {"id": 12, "nombre": "Celta de Vigo",    "ciudad": "Vigo"},
    {"id": 13, "nombre": "RCD Mallorca",     "ciudad": "Palma"},
    {"id": 14, "nombre": "Rayo Vallecano",   "ciudad": "Madrid"},
    {"id": 15, "nombre": "Girona FC",        "ciudad": "Girona"},
    {"id": 16, "nombre": "Alavés",           "ciudad": "Vitoria"},
    {"id": 17, "nombre": "Las Palmas",       "ciudad": "Las Palmas"},
    {"id": 18, "nombre": "Espanyol",         "ciudad": "Barcelona"},
    {"id": 19, "nombre": "Valladolid",       "ciudad": "Valladolid"},
    {"id": 20, "nombre": "Leganés",          "ciudad": "Leganés"},
]

NOMBRES = ["Alejandro","Carlos","Diego","Fernando","Gabriel","Hugo","Iván",
           "Javier","Luis","Marco","Nicolás","Oscar","Pablo","Rafael","Sergio",
           "Tomás","Víctor","Andrés","Borja","Cristian","Marcos","Álvaro",
           "Raúl","Iker","Aitor"]

APELLIDOS = ["García","Martínez","López","Sánchez","Rodríguez","Fernández",
             "González","Torres","Ramírez","Flores","Díaz","Moreno","Jiménez",
             "Ruiz","Herrera","Medina","Castro","Vargas","Romero","Navarro",
             "Iglesias","Santos","Vega","Ramos","Suárez"]

NACIONALIDADES = ["España","Brasil","Argentina","Francia","Portugal",
                  "Alemania","Colombia","Uruguay","Países Bajos","Bélgica"]

def generar_jugadores():
    jugadores = []
    jid = 1
    for eq in EQUIPOS:
        for posicion, cantidad in [("Portero",1),("Defensa",4),("Centrocampista",4),("Delantero",6)]:
            for _ in range(cantidad):
                jugadores.append({
                    "jugador_id":   jid,
                    "nombre":       f"{random.choice(NOMBRES)} {random.choice(APELLIDOS)}",
                    "equipo_id":    eq["id"],
                    "equipo":       eq["nombre"],
                    "posicion":     posicion,
                    "dorsal":       jid % 99 + 1,
                    "nacionalidad": random.choice(NACIONALIDADES),
                })
                jid += 1
    return jugadores

@dataclass
class EventoPartido:
    timestamp:        str
    jornada:          int
    partido_id:       str
    tipo_evento:      str
    minuto:           int
    equipo_local:     str
    equipo_visitante: str
    goles_local:      int
    goles_visitante:  int
    jugador_id:       Optional[int]  = None
    jugador_nombre:   Optional[str]  = None
    equipo_jugador:   Optional[str]  = None
    posicion:         Optional[str]  = None
    tipo_tarjeta:     Optional[str]  = None
    es_penalti:       Optional[bool] = None

print(f"✅ Datos sintéticos cargados | {len(EQUIPOS)} equipos | Listo para generar jugadores")

#  CELDA 3 — CLASE SIMULADOR

In [ ]:
FUERZA_EQUIPOS = {
    "Atlético Madrid": 0.85,
    "Real Madrid":     0.60,
    "FC Barcelona":    0.60,
    "Real Betis":      0.55,
    "Athletic Club":   0.55,
    "Real Sociedad":   0.53,
    "Villarreal CF":   0.53,
    "Rayo Vallecano":  0.25,
}

def prob_marca(local, visitante):
    f_local = FUERZA_EQUIPOS.get(local["nombre"], 0.48)
    f_visit = FUERZA_EQUIPOS.get(visitante["nombre"], 0.48)
    return f_local / (f_local + f_visit)

class SimuladorLiga:

    def __init__(self, jornada, jugadores, delay=0.5):
        self.jornada   = jornada
        self.delay     = delay
        self.producer  = EventHubProducerClient.from_connection_string(
                             CONNECTION_STRING, eventhub_name=EVENT_HUB_NAME)
        self._idx = {}
        for j in jugadores:
            self._idx.setdefault(j["equipo_id"], []).append(j)

    def _jugador(self, equipo_id, excluir_portero=True):
        pool = [j for j in self._idx.get(equipo_id, [])
                if not excluir_portero or j["posicion"] != "Portero"]
        return random.choice(pool) if pool else None

    def _enviar(self, evento: EventoPartido):
        payload = json.dumps(asdict(evento), ensure_ascii=False)
        sufijo  = f"  ⚽ {evento.jugador_nombre}" if evento.tipo_evento == "gol" else (
                  f"  🎯 {evento.jugador_nombre}" if evento.tipo_evento == "asistencia" else (
                  f"  🟨 {evento.jugador_nombre}" if evento.tipo_evento == "tarjeta" else ""))
        print(f"  [{evento.tipo_evento.upper():12}] min {evento.minuto:02d} | "
              f"{evento.equipo_local} {evento.goles_local}-{evento.goles_visitante} "
              f"{evento.equipo_visitante}{sufijo}")
        with self.producer:
            batch = self.producer.create_batch()
            batch.add(EventData(payload))
            self.producer.send_batch(batch)

    def _nuevo_producer(self):
        self.producer = EventHubProducerClient.from_connection_string(
                            CONNECTION_STRING, eventhub_name=EVENT_HUB_NAME)

    def simular_partido(self, local, visitante):
        pid = f"J{self.jornada}_{local['id']}v{visitante['id']}"
        gl, gv = 0, 0
        t0 = datetime.datetime.utcnow()

        print(f"\n{'='*58}")
        print(f"  ⚽  {local['nombre']:20}  vs  {visitante['nombre']}")
        print(f"{'='*58}")

        def ts(m): return (t0 + datetime.timedelta(minutes=m)).isoformat() + "Z"

        def base(tipo, minuto):
            return EventoPartido(
                timestamp=ts(minuto), jornada=self.jornada, partido_id=pid,
                tipo_evento=tipo, minuto=minuto,
                equipo_local=local["nombre"], equipo_visitante=visitante["nombre"],
                goles_local=gl, goles_visitante=gv)

        #  INICIO
        self._nuevo_producer(); self._enviar(base("inicio", 0)); time.sleep(self.delay)

        for minuto in sorted(random.sample(range(1, 91), k=random.randint(5, 15))):
            tipo = random.choices(["gol","tarjeta","ocasion"], weights=[0.35, 0.30, 0.35])[0]

            if tipo == "gol":
                # ── Sesgo por fuerza de equipo ──
                eq_gol = local if random.random() < prob_marca(local, visitante) else visitante
                goleador  = self._jugador(eq_gol["id"])
                asistente = self._jugador(eq_gol["id"])
                if not goleador: continue
                if eq_gol["id"] == local["id"]: gl += 1
                else: gv += 1

                ev = base("gol", minuto)
                ev.jugador_id = goleador["jugador_id"]; ev.jugador_nombre = goleador["nombre"]
                ev.equipo_jugador = eq_gol["nombre"]; ev.posicion = goleador["posicion"]
                ev.es_penalti = random.random() < 0.12
                self._nuevo_producer(); self._enviar(ev); time.sleep(self.delay * 0.4)

                # Asistencia (70%)
                if asistente and asistente["jugador_id"] != goleador["jugador_id"] and random.random() < 0.70:
                    ea = base("asistencia", minuto)
                    ea.jugador_id = asistente["jugador_id"]; ea.jugador_nombre = asistente["nombre"]
                    ea.equipo_jugador = eq_gol["nombre"]; ea.posicion = asistente["posicion"]
                    self._nuevo_producer(); self._enviar(ea); time.sleep(self.delay * 0.3)

            elif tipo == "tarjeta":
                eq_tar = random.choice([local, visitante])
                j = self._jugador(eq_tar["id"])
                if not j: continue
                et = base("tarjeta", minuto)
                et.jugador_id = j["jugador_id"]; et.jugador_nombre = j["nombre"]
                et.equipo_jugador = eq_tar["nombre"]
                et.tipo_tarjeta = random.choices(["amarilla","roja"], weights=[0.85, 0.15])[0]
                self._nuevo_producer(); self._enviar(et)

            time.sleep(self.delay)

        # FINAL
        self._nuevo_producer(); self._enviar(base("final", 90)); time.sleep(self.delay * 0.5)
        print(f"\n  RESULTADO FINAL: {local['nombre']} {gl} – {gv} {visitante['nombre']}\n")
        return {"partido_id": pid, "local": local["nombre"], "visitante": visitante["nombre"],
                "goles_local": gl, "goles_visitante": gv}

    def simular_jornada(self):
        equipos = EQUIPOS.copy()
        random.shuffle(equipos)
        partidos = [(equipos[i], equipos[i+1]) for i in range(0, len(equipos), 2)]

        print(f"\n{'#'*58}")
        print(f"  🏆  JORNADA {self.jornada}  –  {len(partidos)} partidos")
        print(f"{'#'*58}")
        for l, v in partidos:
            print(f"  • {l['nombre']} vs {v['nombre']}")

        resultados = [self.simular_partido(l, v) for l, v in partidos]

        print(f"\n{'#'*58}")
        print(f"  ✅  Jornada {self.jornada} completada — {sum(r['goles_local']+r['goles_visitante'] for r in resultados)} goles enviados")
        print(f"{'#'*58}\n")
        return resultados

print("✅ Clase SimuladorLiga definida")

#  CELDA 4 — EJECUTAR JORNADA  ← RE-EJECUTAR PARA CADA JORNADA

In [ ]:
jugadores   = generar_jugadores()
print(f"✅ {len(jugadores)} jugadores generados para {len(EQUIPOS)} equipos\n")

simulador   = SimuladorLiga(
    jornada   = JORNADA,
    jugadores = jugadores,
    delay     = DELAY_SEGUNDOS,
)
resultados  = simulador.simular_jornada()

# Resumen de resultados en tabla
print("\n📊 RESUMEN DE LA JORNADA:")
print(f"{'Partido':50} {'Resultado':10}")
print("─" * 62)
for r in resultados:
    print(f"  {r['local']:22} {r['goles_local']} – {r['goles_visitante']} {r['visitante']:22}")